# 04. Build Master Manifest

이 노트북에서는 검증이 완료된 **FMA REAL 296곡**과 **Echoes Clean TTA FAKE 3,162곡**을 하나의 통합 manifest로 구성한다.

최종 목표:

```text
FMA REAL 296
      +
Echoes Clean TTA FAKE 3,162
      ↓
master_manifest.csv (3,458 rows)
```

이후 Train / Validation / Test 분할은 **`original_audio` 단위**로 수행하여,
동일한 원곡과 그 원곡에서 파생된 AI 생성물이 서로 다른 split으로 넘어가는 것을 방지한다.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

ECHOES_ROOT = PROJECT_ROOT / "data/raw/Echoes/Echoes"
ECHOES_MANIFEST = ECHOES_ROOT / "dataset_manifest.csv"

FMA_MAPPING = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"
FMA_AUDIO_DIR = PROJECT_ROOT / "data/raw/FMA/selected_30s"

OUTPUT_PATH = PROJECT_ROOT / "data/metadata/master_manifest.csv"

print("PROJECT_ROOT     :", PROJECT_ROOT)
print("ECHOES_MANIFEST  :", ECHOES_MANIFEST)
print("FMA_MAPPING      :", FMA_MAPPING)
print("OUTPUT_PATH      :", OUTPUT_PATH)


## 1. Echoes manifest 로드 및 Clean TTA 재구성

Echoes manifest에서 `type == 'TTA'`인 행만 사용한다.

이전 품질 검증에서 동일한 physical path가 서로 다른 `original_audio`에 연결된 중복 TTA 파일이 확인되었다.
따라서 `path_in_dataset`이 중복된 경우 해당 중복 행을 모두 제외하여 Clean TTA를 구성한다.


In [ ]:
echoes = pd.read_csv(ECHOES_MANIFEST)

print("Echoes manifest rows:", len(echoes))
print("Columns:", echoes.columns.tolist())

tta = echoes[echoes["type"] == "TTA"].copy()

print("\nTTA rows:", len(tta))
print("TTA unique original_audio:", tta["original_audio"].nunique())


In [ ]:
dup_path_mask = tta["path_in_dataset"].duplicated(keep=False)
duplicated_tta = tta[dup_path_mask].copy()

print("Duplicated TTA rows        :", len(duplicated_tta))
print("Duplicated TTA unique paths:", duplicated_tta["path_in_dataset"].nunique())

if len(duplicated_tta):
    display(
        duplicated_tta[
            ["path_in_dataset", "original_audio", "generator", "genre"]
        ].sort_values("path_in_dataset")
    )


In [ ]:
tta_clean = tta[~dup_path_mask].copy().reset_index(drop=True)

print("===== CLEAN TTA =====")
print("Rows                 :", len(tta_clean))
print("Unique original_audio:", tta_clean["original_audio"].nunique())
print("Generators           :", tta_clean["generator"].nunique())

print("\nGenre distribution:")
print(tta_clean["genre"].value_counts())


### Clean TTA 기대값

- Rows: **3,162**
- Unique `original_audio`: **296**
- Generators: **12**


## 2. FMA REAL 매핑 로드

QC가 완료된 최종 FMA REAL mapping을 불러온다.

각 `original_audio`에는 하나의 REAL track만 존재해야 한다.


In [ ]:
real_mapping = pd.read_csv(FMA_MAPPING)
real_mapping["track_id"] = real_mapping["track_id"].astype(int)

print("===== FMA REAL =====")
print("Rows                 :", len(real_mapping))
print("Unique original_audio:", real_mapping["original_audio"].nunique())
print("Unique track_id      :", real_mapping["track_id"].nunique())

display(real_mapping.head())


## 3. REAL과 FAKE의 original_audio 집합 비교

REAL과 FAKE가 동일한 296개의 source family를 공유하는지 확인한다.

이 검사는 이후 `original_audio` 단위 split을 수행하기 위한 핵심 조건이다.


In [ ]:
real_groups = set(real_mapping["original_audio"])
fake_groups = set(tta_clean["original_audio"])

only_real = sorted(real_groups - fake_groups)
only_fake = sorted(fake_groups - real_groups)

print("REAL groups:", len(real_groups))
print("FAKE groups:", len(fake_groups))
print("Only REAL  :", len(only_real))
print("Only FAKE  :", len(only_fake))

if only_real:
    print("\nOnly REAL examples:", only_real[:10])

if only_fake:
    print("\nOnly FAKE examples:", only_fake[:10])

assert real_groups == fake_groups, (
    "REAL과 FAKE의 original_audio 집합이 일치하지 않습니다."
)

print("\noriginal_audio group match: PASS")


## 4. REAL manifest 생성

REAL 데이터는 FMA의 검증 완료된 30초 clip을 사용한다.

- `label = REAL`
- `label_id = 0`
- `source = FMA`
- `generator = real`


In [ ]:
real_manifest = real_mapping.copy()

real_manifest["label"] = "REAL"
real_manifest["label_id"] = 0
real_manifest["source"] = "FMA"
real_manifest["generator"] = "real"
real_manifest["description"] = ""

real_manifest["audio_path"] = real_manifest["track_id"].apply(
    lambda tid: str(
        Path("data/raw/FMA/selected_30s")
        / f"{int(tid) // 1000:03d}"
        / f"{int(tid):06d}.mp3"
    )
)

real_manifest["path_in_dataset"] = pd.NA

display(
    real_manifest[
        [
            "original_audio", "label", "genre", "generator",
            "track_id", "audio_path"
        ]
    ].head()
)


## 5. FAKE manifest 생성

Clean TTA의 모든 AI 생성 음악을 FAKE 데이터로 구성한다.

- `label = FAKE`
- `label_id = 1`
- `source = Echoes`
- `generator`는 실제 생성기 이름 유지


In [ ]:
fake_manifest = tta_clean.copy()

fake_manifest["label"] = "FAKE"
fake_manifest["label_id"] = 1
fake_manifest["source"] = "Echoes"
fake_manifest["track_id"] = pd.NA

fake_manifest["audio_path"] = fake_manifest["path_in_dataset"].apply(
    lambda p: str(Path("data/raw/Echoes/Echoes") / str(p))
)

display(
    fake_manifest[
        [
            "original_audio", "label", "genre", "generator",
            "description", "audio_path"
        ]
    ].head()
)


## 6. REAL + FAKE 통합

두 데이터셋을 공통 schema로 맞춘 뒤 하나의 master manifest로 결합한다.


In [ ]:
master_columns = [
    "original_audio",
    "label",
    "label_id",
    "source",
    "genre",
    "generator",
    "audio_path",
    "track_id",
    "description",
    "path_in_dataset",
]

real_part = real_manifest[master_columns].copy()
fake_part = fake_manifest[master_columns].copy()

master_manifest = pd.concat(
    [real_part, fake_part],
    ignore_index=True,
)

master_manifest.insert(
    0,
    "sample_id",
    [f"sample_{i:05d}" for i in range(len(master_manifest))],
)

print("===== MASTER MANIFEST =====")
print("Rows                 :", len(master_manifest))
print("Unique original_audio:", master_manifest["original_audio"].nunique())

print("\nLabel distribution:")
print(master_manifest["label"].value_counts())

display(master_manifest.head())


### 기대값

```text
Rows                  : 3458
Unique original_audio : 296

FAKE : 3162
REAL : 296
```


## 7. 원곡별 REAL/FAKE 구조 검증

각 `original_audio` group에 REAL이 정확히 1개씩 존재하는지 확인한다.
FAKE는 generator 구성에 따라 원곡별 개수가 다를 수 있다.


In [ ]:
group_check = (
    master_manifest
    .groupby("original_audio")
    .agg(
        total_samples=("sample_id", "size"),
        real_count=("label", lambda x: (x == "REAL").sum()),
        fake_count=("label", lambda x: (x == "FAKE").sum()),
        genre_count=("genre", "nunique"),
        generator_count=("generator", "nunique"),
    )
    .reset_index()
)

bad_real_count = group_check[group_check["real_count"] != 1]
bad_genre_count = group_check[group_check["genre_count"] != 1]

print("Groups                :", len(group_check))
print("REAL count != 1       :", len(bad_real_count))
print("Genre count != 1      :", len(bad_genre_count))

display(group_check.head())


## 8. 실제 오디오 파일 존재 여부 확인

`master_manifest.csv`의 모든 `audio_path`가 실제 파일을 가리키는지 확인한다.


In [ ]:
master_manifest["file_exists"] = master_manifest["audio_path"].apply(
    lambda p: (PROJECT_ROOT / p).exists()
)

missing_files = master_manifest[~master_manifest["file_exists"]].copy()

print("Total rows   :", len(master_manifest))
print("Files exist :", int(master_manifest["file_exists"].sum()))
print("Missing     :", len(missing_files))

if len(missing_files):
    display(
        missing_files[
            ["sample_id", "original_audio", "label", "audio_path"]
        ].head(30)
    )


## 9. 최종 QC

master manifest가 다음 핵심 조건을 모두 만족하는지 확인한다.


In [ ]:
qc_summary = pd.DataFrame({
    "check": [
        "master_rows",
        "unique_original_audio",
        "real_rows",
        "fake_rows",
        "groups_with_real_count_not_1",
        "groups_with_genre_count_not_1",
        "missing_audio_files",
        "duplicate_sample_id",
    ],
    "value": [
        len(master_manifest),
        master_manifest["original_audio"].nunique(),
        int((master_manifest["label"] == "REAL").sum()),
        int((master_manifest["label"] == "FAKE").sum()),
        len(bad_real_count),
        len(bad_genre_count),
        len(missing_files),
        int(master_manifest["sample_id"].duplicated().sum()),
    ],
})

display(qc_summary)

core_qc_pass = (
    len(master_manifest) == 3458
    and master_manifest["original_audio"].nunique() == 296
    and int((master_manifest["label"] == "REAL").sum()) == 296
    and int((master_manifest["label"] == "FAKE").sum()) == 3162
    and len(bad_real_count) == 0
    and len(bad_genre_count) == 0
    and len(missing_files) == 0
    and int(master_manifest["sample_id"].duplicated().sum()) == 0
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", core_qc_pass)


## 10. master_manifest.csv 저장

최종 QC가 통과한 manifest를 저장한다.


In [ ]:
if not core_qc_pass:
    raise RuntimeError(
        "Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요."
    )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

master_manifest.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_PATH)
print("Rows :", len(master_manifest))


## 다음 단계

`master_manifest.csv`가 완성되면 다음 단계에서는 **`original_audio` 단위 Train / Validation / Test split**을 생성한다.

핵심 원칙:

```text
같은 original_audio
   ├─ REAL 1곡
   └─ FAKE 여러 곡

→ 모두 반드시 동일한 split에 배정
```

이 원칙을 지켜야 원곡 정보가 Train과 Test 사이에 누출되는 source-family leakage를 방지할 수 있다.
